# Bukhari Pilot EDA\n\nThis notebook explores the 250-record `eng-bukhari` pilot dataset generated by `scripts/build_bukhari_pilot.py`.

In [ ]:
from pathlib import Path\nimport json\nimport re\nfrom collections import Counter\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nplt.style.use('seaborn-v0_8-whitegrid')\npd.set_option('display.max_columns', 200)

In [ ]:
ROOT = Path('..')\nocc_path = ROOT / 'data/interim/bukhari_eng_pilot.jsonl'\nsearch_path = ROOT / 'data/curated/bukhari_eng_pilot.search.jsonl'\nsnapshot_path = ROOT / 'data/raw/source_snapshot.bukhari_eng.json'\n\nocc = pd.read_json(occ_path, lines=True)\nsearch_docs = pd.read_json(search_path, lines=True)\n\nwith open(snapshot_path, 'r', encoding='utf-8') as f:\n    snapshot = json.load(f)\n\nprint('Occurrences:', occ.shape)\nprint('Search docs:', search_docs.shape)\nprint('Snapshot edition:', snapshot['edition_id'])\nprint('Retrieved at:', snapshot['retrieved_at'])

In [ ]:
occ.head(3)

In [ ]:
summary = pd.DataFrame({\n    'rows': [len(occ)],\n    'unique_ids': [occ['id'].nunique()],\n    'unique_hadith_numbers': [occ['hadith_number'].nunique()],\n    'unique_books': [occ['book_ref'].nunique()],\n    'duplicate_ids': [occ['id'].duplicated().sum()],\n})\nsummary

In [ ]:
null_rate = (occ.isna().mean() * 100).sort_values(ascending=False).round(2)\nnull_rate.to_frame('null_percent')

In [ ]:
book_counts = occ['book_ref'].value_counts().sort_index(key=lambda s: s.astype(int))\n\nax = book_counts.plot(kind='bar', figsize=(12, 4), color='#2f6db3')\nax.set_title('Records per book_ref (pilot sample)')\nax.set_xlabel('book_ref')\nax.set_ylabel('record count')\nplt.tight_layout()\nplt.show()\n\nbook_counts.head(20)

In [ ]:
occ['text_length_chars'] = occ['translation_en'].str.len()\nocc['text_length_words'] = occ['translation_en'].str.split().str.len()\n\nfig, axes = plt.subplots(1, 2, figsize=(13, 4))\nocc['text_length_chars'].plot(kind='hist', bins=30, ax=axes[0], color='#4C9F70')\naxes[0].set_title('Text length (characters)')\n\nocc['text_length_words'].plot(kind='hist', bins=30, ax=axes[1], color='#A05EB5')\naxes[1].set_title('Text length (words)')\n\nplt.tight_layout()\nplt.show()\n\nocc[['text_length_chars', 'text_length_words']].describe().T

In [ ]:
grade_counts = Counter()\nfor claims in occ['grade_claims']:\n    if isinstance(claims, list):\n        for c in claims:\n            grade = (c or {}).get('grade')\n            if grade:\n                grade_counts[grade] += 1\n\npd.DataFrame(grade_counts.items(), columns=['grade', 'count']).sort_values('count', ascending=False)

In [ ]:
STOPWORDS = {\n    'the','and','to','of','in','a','is','for','that','on','with','as','be','he','it',\n    'his','i','was','are','at','or','from','by','an','this','who','said','allah','messenger'\n}\n\ntokens = []\nfor text in occ['translation_en'].fillna(''):\n    words = re.findall(r"[a-zA-Z']+", text.lower())\n    tokens.extend([w for w in words if w not in STOPWORDS and len(w) > 2])\n\ntop_terms = pd.DataFrame(Counter(tokens).most_common(30), columns=['term', 'count'])\ntop_terms

In [ ]:
queries = json.loads((ROOT / 'tests/queries/pilot_queries.json').read_text(encoding='utf-8'))\n\ndef naive_hits(q: str) -> int:\n    q_terms = [t for t in re.findall(r"[a-zA-Z']+", q.lower()) if t]\n    if not q_terms:\n        return 0\n    mask = occ['translation_en'].str.lower().apply(lambda t: all(term in t for term in q_terms))\n    return int(mask.sum())\n\nquery_hits = pd.DataFrame({\n    'query': queries,\n    'naive_exact_term_hits': [naive_hits(q) for q in queries]\n}).sort_values('naive_exact_term_hits', ascending=False)\n\nquery_hits